# Reproducibility notebook (Google Colab)

This notebook rebuilds the project end-to-end (data fetch → analysis → figures/tables → optional DOCX).

- Expected runtime: ~10–30 minutes (depends on network + Colab load).
- Output directories: `results/` and `plots/publication/`.

If you opened this notebook outside the repo root, run the **Clone / mount repo** cell first.

## 0) Clone / mount repo (only if needed)

Option A (recommended for sharing): upload a ZIP of the repo to Colab and unzip it.

Option B: clone from a public Git repository (fill `REPO_URL`).

In [ ]:
# If you already opened this notebook inside the repo, you can skip this cell.

import os
from pathlib import Path

REPO_URL = ""  # e.g., "https://github.com/<org>/<repo>.git" (leave empty if not cloning)
REPO_DIR = "lnrhxdyj"

root = Path.cwd()
if (root / "scripts" / "run_all.sh").exists():
    print("Repo root detected:", root)
elif REPO_URL:
    !git clone --depth 1 $REPO_URL $REPO_DIR
    os.chdir(REPO_DIR)
    print("Cloned into:", Path.cwd())
else:
    raise RuntimeError(
        "Repo root not detected. Either open this notebook from the repo root, "
        "or set REPO_URL to a public git URL and re-run this cell."
    )


## 1) Install dependencies

Colab’s Python version may differ from the local development environment.
We install a small set of runtime dependencies used by the scripts.

In [ ]:
!python -V
!pip -q install -r requirements-colab.txt


## 2) Run the full pipeline

Set the cohort profile:
- `older_adult_65plus`: 65+ only (ED signals available)
- `older_adult_strata`: 65–74 / 75–84 / 85+ (no ED proxying; RESP-NET strata only)

By default this will fetch fresh public snapshots. To rerun without fetching, set `SKIP_FETCH = \"1\"`.

In [ ]:
COHORT_PROFILE = "older_adult_strata"  # change to "older_adult_65plus" if needed
SKIP_FETCH = "0"  # set to "1" to reuse existing snapshots under results/

import os
os.environ["COHORT_PROFILE"] = COHORT_PROFILE
os.environ["SKIP_FETCH"] = SKIP_FETCH

!bash scripts/run_all.sh


## 3) Quick sanity check: key figures exist

In [ ]:
from pathlib import Path
from IPython.display import Image, display
import os
import subprocess

subprocess.check_call([
    "python3",
    "tools/smoke_check_outputs.py",
    "--cohort-profile",
    os.environ.get("COHORT_PROFILE", ""),
])

fig_dir = Path("plots/publication")
candidates = [
    fig_dir / "Fig1_time_series_overview.png",
    fig_dir / "Fig3_paired_benchmark.png",
    fig_dir / "FigS4_incremental_value_beyond_seasonality.png",
]

for p in candidates:
    print(f"{p}: {'OK' if p.exists() else 'MISSING'}")

for p in candidates:
    if p.exists():
        display(Image(filename=str(p)))
        break


## 4) Optional: rebuild manuscript DOCX

This step uses Pandoc + `python-docx` to produce a submission-style `.docx`.

## 5) Optional: package outputs

Creates a single ZIP suitable for sharing with collaborators.

In [ ]:
!rm -f colab_repro_outputs.zip
!zip -r colab_repro_outputs.zip results plots/publication manuscript_colab.docx -x "results/_legacy/*" "results/**/_legacy/*"
!ls -lh colab_repro_outputs.zip
